In [ ]:
# 검색 API의 하루 호출 한도는 25,000회
# 한달 호출 한도는 775,000 회
#https://api.ncloud-docs.com/docs/naver-api-hub-search-news#요청-쿼리-파라미터

In [ ]:
# NAVER NEWS 검색 결과 원본 + 메타 데이터를 합쳐서 JSONL로 저장하기 위한 파일을 만들어내는 함수
# 검색어, 원하는 날짜를 입력하면 검색 결과 중 원하는 날짜에 해당하는 것만 검색 결과로 받음
# 날짜를 비워두면 검색 결과 전체를 받음
import os
import json
import requests

from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

NAVER_CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
NAVER_CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")

NAVER_NEWS_URL = "https://naverapihub.apigw.ntruss.com/search/v1/news"


def parse_pubdate(pub_date: str):
    """
    네이버 뉴스 API의 pubDate 문자열을 datetime 객체로 변환합니다.
    """
    return datetime.strptime(
        pub_date,
        "%a, %d %b %Y %H:%M:%S %z"
    )


def search_naver_news(
    query: str,
    target_date: str | None = None
):
    """
    네이버 뉴스 검색 API를 이용해 뉴스를 검색합니다.

    Parameters
    ----------
    query : str
        검색어

    target_date : str | None
        "YYYY-MM-DD" 형식의 날짜
        None 또는 ""이면 날짜 필터링을 하지 않습니다.

    Returns
    -------
    list[dict]
        메타데이터가 포함된 flat 구조의 뉴스 데이터
    """

    if not query or not query.strip():
        raise ValueError("검색어(query)는 비워둘 수 없습니다.")

    query = query.strip()

    # "" → None
    target_date = target_date.strip() if target_date else None

    if target_date:
        try:
            target = datetime.strptime(
                target_date,
                "%Y-%m-%d"
            ).date()
        except ValueError:
            raise ValueError(
                "날짜는 YYYY-MM-DD 형식이어야 합니다. "
                "예: 2026-10-13"
            )
    else:
        target = None

    if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:
        raise ValueError(
            "NAVER_CLIENT_ID 또는 NAVER_CLIENT_SECRET이 설정되지 않았습니다."
        )

    headers = {
        "X-NCP-APIGW-API-KEY-ID": NAVER_CLIENT_ID,
        "X-NCP-APIGW-API-KEY": NAVER_CLIENT_SECRET,
    }

    display = 100
    start = 1
    sort = "date"

    results = []

    while start <= 1000:

        params = {
            "query": query,
            "display": display,
            "start": start,
            "sort": sort,
            "format": "json",
        }

        response = requests.get(
            NAVER_NEWS_URL,
            headers=headers,
            params=params,
            timeout=10,
        )

        response.raise_for_status()

        data = response.json()

        items = data.get("items", [])

        if not items:
            break

        # 해당 API 요청이 이루어진 시각
        collected_at = datetime.now(
            ZoneInfo("Asia/Seoul")
        ).isoformat()

        stop_search = False

        for item in items:

            article_date = parse_pubdate(
                item["pubDate"]
            ).date()

            # 날짜가 지정되어 있는 경우 필터링
            if target is not None:

                if article_date > target:
                    continue

                elif article_date < target:
                    stop_search = True
                    break

            # flat JSONL record
            record = {
                # -------------------------
                # 우리가 추가하는 메타데이터
                # -------------------------
                "source": "naver_news_api",
                "query": query,
                "target_date": target_date,
                "sort": sort,
                "collected_at": collected_at,

                # -------------------------
                # API 요청 / 응답 메타데이터
                # -------------------------
                "request_start": start,
                "requested_display": display,

                "api_lastBuildDate": data.get("lastBuildDate"),
                "api_total": data.get("total"),
                "api_start": data.get("start"),
                "api_display": data.get("display"),

                # -------------------------
                # 뉴스 기사 데이터
                # -------------------------
                "title": item.get("title"),
                "originallink": item.get("originallink"),
                "link": item.get("link"),
                "description": item.get("description"),
                "pubDate": item.get("pubDate"),
            }

            results.append(record)

        if stop_search:
            break

        # 전체 검색 결과를 모두 확인했다면 종료
        if start + display > data.get("total", 0):
            break

        start += display

    return results

In [10]:
#검색 결과를 JSONL로 저장하는 함수
def save_news_jsonl(
    news: list[dict],
    output_path: str | Path
):
    """
    뉴스 데이터를 JSONL 파일로 저장합니다.
    """

    output_path = Path(output_path)

    # 상위 폴더가 없으면 생성
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with output_path.open(
        "w",
        encoding="utf-8"
    ) as f:

        for record in news:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )

    print(f"저장 완료: {output_path}")
    print(f"저장 기사 수: {len(news):,}개")

In [14]:
news = search_naver_news(
    query="삼성전자",
    target_date="2026-10-12"
)

save_news_jsonl(
    news=news,
    output_path="news/raw/naver_news_삼성전자_2026-10-12.jsonl"
)

저장 완료: news/raw/naver_news_삼성전자_2026-10-12.jsonl
저장 기사 수: 0개


In [16]:
news = search_naver_news(
    query="삼성전자"
)

save_news_jsonl(
    news=news,
    output_path="news/raw/naver_news_삼성전자.jsonl"
)

저장 완료: news/raw/naver_news_삼성전자.jsonl
저장 기사 수: 1,000개


In [25]:
news = search_naver_news(
    query="경상북도 경주 동천동"
)

save_news_jsonl(
    news=news,
    output_path="news/raw/naver_news_경상북도_경주_동천동.jsonl"
)

저장 완료: news/raw/naver_news_경상북도_경주_동천동.jsonl
저장 기사 수: 461개


기존의 일반적인 스크래핑 도구(예: BeautifulSoup)가 HTML 전체를 가져와 개발자가 직접 규칙을 정해야 하는 반면, Trafilatura는 구조 분석과 휴리스틱(규칙 기반 감지) 알고리즘을 사용해 자동으로 본문을 찾아냅니다.


1. 웹페이지 다운로드 및 전처리 (Fetch & Preprocess)

• HTML 수집: 대상 URL에서 HTML 소스 코드를 다운로드합니다. 이 과정에서 빠른 속도를 위해 내부적으로 urllib3나 requests 라이브러리를 활용합니다.
• 디코딩 및 인코딩 복원: 웹페이지의 문자 인코딩(UTF-8, EUC-KR 등)을 올바르게 감지하여 깨진 글자가 없도록 텍스트를 복원합니다.
• 불필요한 태그 일차 제거: 본문 분석에 전혀 방해가 되는 스크립트(<script>), 스타일시트(<style>), 주석(<!-- comment -->) 태그를 먼저 파싱 트리에서 완전히 삭제합니다.

2. 웹페이지 구조 트리(LXML Parsing) 생성

• 다운로드한 HTML을 고속 XML/HTML 처리 라이브러리인 lxml을 사용하여 DOM(Document Object Model) 트리 구조로 변환합니다. 이를 통해 웹페이지의 부모-자식 태그 관계를 메모리 상에 정밀하게 구조화합니다.


3. 휴리스틱 기반 본문 감지 (Heuristic Text Detection)

Trafilatura의 핵심 알고리즘이 작동하는 단계입니다. 단순히 특정 태그를 찾는 것이 아니라, 텍스트의 밀도와 태그의 속성을 분석합니다.
• 텍스트/링크 비율(Link Density) 계산: 특정 구역(디브 블록 등)에 링크( 태그)가 너무 많다면, 그곳은 본문이 아니라 메뉴나 광고 영역일 확률이 높다고 판단하여 제외합니다.
• 블록 요소 분석: <div>, <article>, <p> 등의 블록 태그 내부의 글자 수(텍스트 밀도)를 측정합니다. 글자 수가 길고 연속적인 문장이 많은 곳을 본문 후보군으로 지정합니다.
• 부정적인 키워드 필터링: 클래스명이나 ID에 sidebar, footer, comment, nav, share 등의 단어가 포함된 태그는 본문이 아닐 가능성이 크므로 가중치를 낮추거나 제거합니다.

4. 메타데이터 및 언어 추출 (Metadata & Language Detection)

• 메타데이터 수집: 본문 외에도 기사 제목, 저자, 발행일, 웹사이트 이름, 키워드 등의 메타데이터를 HTML 헤더(<head>) 및 Open Graph 태그 등에서 추출합니다.
• 언어 감지: 추출된 텍스트가 어떤 언어로 작성되었는지 자동으로 감지하여 필터링이나 형태소 분석의 기반을 마련합니다.

5. 구조화된 출력 (Structured Output Generation)

사용자가 원하는 포맷에 맞춰 최종 데이터를 변환하여 반환합니다.
• 형식 지원: 순수 텍스트(Plain Text)뿐만 아니라, 글의 구조(제목, 단락, 인용구 등)를 유지하는 Markdown, XML, JSON 형식으로 출력이 가능합니다.

In [26]:
#pip install trafilatura requests

In [27]:
import json
import time
import requests

from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

from trafilatura import bare_extraction


def crawl_news_jsonl(
    input_path: str | Path,
    output_path: str | Path,
    request_delay: float = 1.0,
):
    """
    네이버 뉴스 API 검색 결과 JSONL을 읽어서
    각 기사의 originallink에 접속한 뒤,

    - 실제 기사 제목
    - 기자명
    - 기사 날짜
    - 언론사명
    - 기사 본문

    등을 Trafilatura로 추출하여
    새로운 flat JSONL 파일로 저장합니다.

    Parameters
    ----------
    input_path : str | Path
        네이버 뉴스 API 검색 결과 JSONL 경로

    output_path : str | Path
        크롤링 결과를 저장할 JSONL 경로

    request_delay : float
        각 기사 요청 사이의 대기 시간(초)
    """

    input_path = Path(input_path)
    output_path = Path(output_path)

    if not input_path.exists():
        raise FileNotFoundError(
            f"입력 파일을 찾을 수 없습니다: {input_path}"
        )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    session = requests.Session()

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 "
            "(Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/140.0 Safari/537.36"
        )
    })

    success_count = 0
    fail_count = 0

    with (
        input_path.open(
            "r",
            encoding="utf-8"
        ) as input_file,

        output_path.open(
            "w",
            encoding="utf-8"
        ) as output_file
    ):

        for line_number, line in enumerate(
            input_file,
            start=1
        ):

            line = line.strip()

            if not line:
                continue

            # -----------------------------
            # 1. 기존 API 데이터 읽기
            # -----------------------------
            record = json.loads(line)

            original_link = record.get(
                "originallink"
            )

            # -----------------------------
            # 2. 크롤링용 필드 초기화
            # -----------------------------
            record["crawled_at"] = datetime.now(
                ZoneInfo("Asia/Seoul")
            ).isoformat()

            record["crawl_status"] = None
            record["crawl_http_status"] = None
            record["crawl_final_url"] = None
            record["crawl_error"] = None

            record["crawled_title"] = None
            record["crawled_author"] = None
            record["crawled_date"] = None
            record["crawled_sitename"] = None

            record["content"] = None
            record["content_length"] = 0

            # -----------------------------
            # 3. originallink 확인
            # -----------------------------
            if not original_link:

                record["crawl_status"] = "failed"
                record["crawl_error"] = (
                    "originallink가 없습니다."
                )

                fail_count += 1

            else:

                try:

                    # -----------------------------
                    # 4. 원문 페이지 요청
                    # -----------------------------
                    response = session.get(
                        original_link,
                        timeout=15,
                        allow_redirects=True
                    )

                    record["crawl_http_status"] = (
                        response.status_code
                    )

                    record["crawl_final_url"] = (
                        response.url
                    )

                    response.raise_for_status()

                    # -----------------------------
                    # 5. Trafilatura로 기사 추출
                    # -----------------------------
                    document = bare_extraction(
                        response.content,
                        url=response.url,
                        with_metadata=True,
                        include_comments=False,
                        include_tables=False,
                    )

                    # -----------------------------
                    # 6. 추출 실패
                    # -----------------------------
                    if document is None:

                        record["crawl_status"] = (
                            "failed"
                        )

                        record["crawl_error"] = (
                            "Trafilatura가 "
                            "기사 데이터를 추출하지 못했습니다."
                        )

                        fail_count += 1

                    # -----------------------------
                    # 7. 추출 성공
                    # -----------------------------
                    else:

                        record["crawled_title"] = (
                            document.title
                        )

                        record["crawled_author"] = (
                            document.author
                        )

                        record["crawled_date"] = (
                            document.date
                        )

                        record["crawled_sitename"] = (
                            document.sitename
                        )

                        content = document.text

                        if content:
                            content = content.strip()

                        # 본문이 실제로 있는지 확인
                        if content:

                            record["content"] = content

                            record["content_length"] = (
                                len(content)
                            )

                            record["crawl_status"] = (
                                "success"
                            )

                            success_count += 1

                        else:

                            record["crawl_status"] = (
                                "failed"
                            )

                            record["crawl_error"] = (
                                "기사 메타데이터는 추출했지만 "
                                "본문을 추출하지 못했습니다."
                            )

                            fail_count += 1

                # -----------------------------
                # HTTP 관련 오류
                # -----------------------------
                except requests.RequestException as e:

                    record["crawl_status"] = "failed"

                    record["crawl_error"] = (
                        f"{type(e).__name__}: {e}"
                    )

                    fail_count += 1

                # -----------------------------
                # 기타 오류
                # -----------------------------
                except Exception as e:

                    record["crawl_status"] = "failed"

                    record["crawl_error"] = (
                        f"{type(e).__name__}: {e}"
                    )

                    fail_count += 1

            # -----------------------------
            # 8. JSONL 한 줄 저장
            # -----------------------------
            output_file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )

            # -----------------------------
            # 9. 진행 상황 출력
            # -----------------------------
            print(
                f"[{line_number}] "
                f"{record['crawl_status']} | "
                f"{record.get('crawled_title') or record.get('title', '')}"
            )

            # 서버에 너무 빠르게 요청하지 않도록 대기
            time.sleep(request_delay)

    print()
    print("크롤링 완료")
    print(f"저장 위치: {output_path}")
    print(f"성공: {success_count:,}개")
    print(f"실패: {fail_count:,}개")

In [28]:
crawl_news_jsonl(
    input_path="news/raw/naver_news_경상북도_경주_동천동.jsonl",
    output_path="news/crawled/naver_news_경상북도_경주_동천동.jsonl"
)

[1] success | [패트롤]경주시-달서구-수성구-대구대-영남이공대-경북관광공사
[2] success | ‘맛슐랭 가이드’ 경주 동천동 뒷고기·된장국수 고깃집 “항아리삼겹살의 특별함”…양평 항정살수육·강된장연어쌈말이 식당 위치는?
[3] success | [패트롤] 경주시-영천시-대구 달서구-수성구-경북문화관광공사-영남이공대
[4] success | 이동협 경주시의회 의장, 경북도의원 출마 선언…"경북도-경주시 잇는 강력한 엔진 될 것"
[5] success | 경주·경산 광역의원 1석씩 증원... 선거구 대격변에 기초의원 예비후보 '당혹'
[6] success | [대구일보 6.3 지방선거 여론조사] 주낙영 38.6% vs 박병훈 36.7%…‘초박빙’ 접전
[7] success | 주낙영·박병훈 오차 내 접전…본선보다 뜨거운 경주 경선
[8] success | 영남일보
[9] success | 경주 3개전서 춘향대제 봉행…"전통 제례문화 계승"
[10] success | 경주 3전(三殿)서 춘향대제 봉행
[11] success | [E-로컬뉴스] 경주시, 포항시, 경주시의회, 달서구, 계명대, 대구보건대 소식
[12] success | 경주시 동천동·보덕동 현장소통마당 개최로 실무 중심 민원 해결 나서
[13] success | 경주 택시 기본요금 4천500원으로 인상…다음 달부터 적용
[14] success | 경주시, 동천동·보덕동서 ‘현장소통마당’ 열고 주민 목소리 청취
[15] success | 경주시, 시민과 함께 만드는 현장소통마당
[16] success | [위클리오늘] 주낙영 경주시장, 답은 현장에 ... 동천동.보덕동서 주민 목소리 직접 청취
[17] success | 경주시, 동천동·보덕동서 현장소통 이어가...주민 목소리 직접 청취
[18] success | 경주시, 동천동·보덕동서 현장소통 이어가…주민 목소리 직접 청취
[19] success | ≪브레이크뉴스≫ 경주시, 동천·보덕동서 ‘현장소통마당’…주민 목소리 직접 청취
[20]

In [29]:
import json
from pathlib import Path


def print_failed_articles(jsonl_path: str | Path):
    jsonl_path = Path(jsonl_path)

    failed_count = 0

    with jsonl_path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            if record.get("crawl_status") != "failed":
                continue

            failed_count += 1

            title = (
                record.get("crawled_title")
                or record.get("title")
                or "(제목 없음)"
            )

            print(f"[{failed_count}] JSONL line: {line_number}")
            print(f"제목: {title}")
            print(f"URL: {record.get('originallink')}")
            print(f"HTTP 상태: {record.get('crawl_http_status')}")
            print(f"실패 사유: {record.get('crawl_error')}")
            print("-" * 80)

    print(f"\n총 실패 기사 수: {failed_count:,}개")

In [30]:
print_failed_articles(
    jsonl_path='news/crawled/naver_news_경상북도_경주_동천동.jsonl'
)

[1] JSONL line: 69
제목: 한수원, APEC 성공 개최 '통 큰 지원'
URL: https://www.idaegu.co.kr/news/articleView.html?idxno=513316
HTTP 상태: None
실패 사유: ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
--------------------------------------------------------------------------------
[2] JSONL line: 103
제목: <b>경주</b>시, ‘숭덕전, 숭혜전, 숭신전’ 3전 춘향대제 봉행
URL: http://www.metroseoul.co.kr/article/20250320500626
HTTP 상태: None
실패 사유: ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
--------------------------------------------------------------------------------
[3] JSONL line: 133
제목: [인사] 하나은행 ; 산업통상자원부 ; 통일부 등
URL: https://www.hankyung.com/society/article/2023011663331
HTTP 상태: 403
실패 사유: HTTPError: 403 Client Error: Forbidden for url: https://www.hankyung.com/society/article/2023011663331
--------------------------------------------------------------------------------
[4] JSONL line: 183
제목: 명절 앞